# 🚀 Đào tạo Siêu Kiến Trúc Model E (Ultimate VQA-E)

Notebook này chứa toàn bộ quy trình Master Blueprint để đào tạo Model E từ đầu đến cuối trên máy cục bộ của bạn. Bao gồm 4 Phase chính.

- CPU: Intel Core i7-14700KF (20 Cores - 12 Workers DataLoader là tối ưu)
- GPU: NVIDIA GeForce RTX 5070 Ti (16GB VRAM)
- Model: Model E (CLIP ViT-B/32 + FiLM + Linear Projections + LSTM)

In [1]:
# Kiểm tra sức mạnh GPU trước khi bắt đầu
!nvidia-smi

Sat Mar 14 01:00:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.48.01              Driver Version: 590.48.01      CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5070 Ti     Off |   00000000:01:00.0  On |                  N/A |
|  0%   51C    P3             36W /  300W |    3982MiB /  16303MiB |      3%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Phase 1: Base Training (15 Epochs)

Giai đoạn này sẽ train toàn bộ khối LSTM Decoder, FiLM, và Projections. Image Encoder (CLIP ViT) sẽ bị đóng băng hoàn toàn.

* **LR**: `1e-3` (với 3 epoch Warmup từ `1e-4` → `1e-3`, sau đó Cosine Decay)
* **Batch Size**: `128` (Có thể đẩy lên 256 nếu RAM/VRAM cho phép)
* **Workers**: `12` (Tối đa hóa sức mạnh của i7-14700KF)

In [3]:
!python src/train.py --model E \
    --epochs 15 \
    --batch_size 256 \
    --num_workers 12 \
    --lr 1e-3 \
    --warmup_epochs 3

Train: 181298 | Val: 88488
Loading weights: 100%|█████████████████████| 199/199 [00:00<00:00, 41035.72it/s]
CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0.

In [5]:
!python src/train.py --model E \
    --epochs 20 \
    --batch_size 256 \
    --num_workers 16 \
    --lr 1e-3 \
    --warmup_epochs 3 \
    --resume checkpoints/model_e_resume.pth


Train: 181298 | Val: 88488
Loading weights: 100%|█████████████████████| 199/199 [00:00<00:00, 83525.12it/s]
CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0.

## Phase 2: Coverage & Fine-Tuning Vision (10 Epochs)

Giai đoạn này tiếp tục lấy Checkpoint tốt nhất từ Phase 1. Mở khóa (Unfreeze) block cuối cùng của mô hình CLIP ViT để tinh chỉnh nó cho task VQA-E, và bật cơ chế `Coverage` chống lặp từ.

- **LR**: Base `1e-4` (Thấp hơn để tinh chỉnh), CNN LR = Base * 0.1
- **Resume**: Nạp lại weights và SequentialLR states để học mượt mà
- **Coverage**: `True` (lambda = 1.0)

In [ ]:
!python src/train.py --model E \
    --epochs 10 \
    --batch_size 256 \
    --num_workers 16 \
    --lr 1e-4 \
    --warmup_epochs 0 \
    --resume checkpoints/model_e_best.pth \
    --finetune_cnn \
    --cnn_lr_factor 0.1 \
    --coverage \
    --coverage_lambda 1.0

## Phase 3: Scheduled Sampling (5 Epochs)

Bật cơ chế nhả chữ giả lập để giảm Exposure Bias. Decoder lúc này thay vì được đút bộ từ khóa thực tế 100%, nó sẽ từ từ học cách sống chung với những dự báo quá khứ của chính nó.

In [ ]:
!python src/train.py --model E \
    --epochs 5 \
    --batch_size 256 \
    --num_workers 16 \
    --lr 5e-5 \
    --warmup_epochs 0 \
    --resume checkpoints/model_e_best.pth \
    --scheduled_sampling \
    --ss_k 5.0 \
    --coverage \
    --coverage_lambda 1.0

## Phase 4: Self-Critical Sequence Training (RL - Tùy chọn cực mạnh)

Sử dụng thuật toán REINFORCE của Học tăng cường. Trực tiếp chọc tay vào thước đo `BLEU-4` thay vì Cross-Entropy thụ động. Giai đoạn này rất nặng nhưng mang tính quyết định cho điểm số cuối.

In [ ]:
!python src/train_rl.py \
    --epochs 3 \
    --batch_size 128 \
    --lr 1e-5 \
    --resume checkpoints/model_e_best.pth